# Charging Baseline Parsing – Version 1

## Objective
Parse the official charging-point source, convert it into a clean station-level dataset, and determine which charging stations belong to the interurban baseline by matching them to the mobility backbone.

## Inputs
- Official charging dataset (`export_mensual_mat_202602.zip`)
- Mobility backbone (`mobility_backbone.gpkg`)

## Outputs
- Clean charging-stations table
- Charging stations GeoDataFrame
- Interurban charging baseline matched to the mobility backbone

In [1]:
import requests

url = "https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml"

response = requests.get(url)
xml_content = response.content

print(xml_content[:1000].decode('utf-8', errors='ignore'))

<?xml version="1.0" encoding="UTF-8"?><d2:payload xmlns:d2="http://datex2.eu/schema/3/d2Payload" xsi:type="egi:EnergyInfrastructureTablePublication" lang="es" modelBaseVersion="3" xmlns:com="http://datex2.eu/schema/3/common" xmlns:loc="http://datex2.eu/schema/3/locationReferencing" xmlns:roa="http://datex2.eu/schema/3/roadTrafficData" xmlns:egi="http://datex2.eu/schema/3/energyInfrastructure" xmlns:prk="http://datex2.eu/schema/3/parking" xmlns:comx="http://datex2.eu/schema/3/commonExtension" xmlns:fac="http://datex2.eu/schema/3/facilities" xmlns:locx="http://datex2.eu/schema/3/locationExtension" xmlns:vms="http://datex2.eu/schema/3/vms" xmlns:sit="http://datex2.eu/schema/3/situation" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
    <com:feedDescription>
        <com:values>
            <com:value lang="es">Publicación de electrolineras</com:value>
        </com:values>
    </com:feedDescription>
    <com:publicationTime>2026-03-28T10:16:07.617+01:00</com:publicationTime>
   


In [2]:
import xml.etree.ElementTree as ET

root = ET.fromstring(xml_content)

print(root.tag)

{http://datex2.eu/schema/3/d2Payload}payload


In [3]:
for child in root:
    print(child.tag)

{http://datex2.eu/schema/3/common}feedDescription
{http://datex2.eu/schema/3/common}publicationTime
{http://datex2.eu/schema/3/common}publicationCreator
{http://datex2.eu/schema/3/energyInfrastructure}headerInformation
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureTable


In [4]:
for elem in root.iter():
    print(elem.tag)
    break

{http://datex2.eu/schema/3/d2Payload}payload


In [5]:
for elem in root.iter():
    if "energyInfrastructureTable" in elem.tag:
        table = elem
        break

# inspect children
for child in table:
    print(child.tag)

{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
{http://datex2.eu/schema/3/energyInfrastructure}ene

In [6]:
# find the first energyInfrastructureSite
site = None
for elem in root.iter():
    if elem.tag.endswith('energyInfrastructureSite'):
        site = elem
        break

print(site.tag)
print("children:\n")

for child in site:
    print(child.tag)

{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
children:

{http://datex2.eu/schema/3/facilities}name
{http://datex2.eu/schema/3/facilities}lastUpdated
{http://datex2.eu/schema/3/facilities}accessibility
{http://datex2.eu/schema/3/facilities}operatingHours
{http://datex2.eu/schema/3/facilities}locationReference
{http://datex2.eu/schema/3/facilities}operator
{http://datex2.eu/schema/3/energyInfrastructure}typeOfSite
{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureStation


In [7]:
def print_tree(elem, level=0, max_depth=3):
    if level > max_depth:
        return
    indent = "  " * level
    print(f"{indent}{elem.tag}")
    for child in elem:
        print_tree(child, level + 1, max_depth)

print_tree(site, max_depth=3)

{http://datex2.eu/schema/3/energyInfrastructure}energyInfrastructureSite
  {http://datex2.eu/schema/3/facilities}name
    {http://datex2.eu/schema/3/common}values
      {http://datex2.eu/schema/3/common}value
  {http://datex2.eu/schema/3/facilities}lastUpdated
  {http://datex2.eu/schema/3/facilities}accessibility
  {http://datex2.eu/schema/3/facilities}operatingHours
    {http://datex2.eu/schema/3/facilities}label
    {http://datex2.eu/schema/3/facilities}overallPeriod
      {http://datex2.eu/schema/3/common}overallStartTime
  {http://datex2.eu/schema/3/facilities}locationReference
    {http://datex2.eu/schema/3/locationReferencing}_locationReferenceExtension
      {http://datex2.eu/schema/3/locationReferencing}facilityLocation
    {http://datex2.eu/schema/3/locationReferencing}coordinatesForDisplay
      {http://datex2.eu/schema/3/locationReferencing}latitude
      {http://datex2.eu/schema/3/locationReferencing}longitude
  {http://datex2.eu/schema/3/facilities}operator
    {http://dat

In [8]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import math

url = "https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml"
xml_content = requests.get(url).content
root = ET.fromstring(xml_content)

NS = {
    "d2": "http://datex2.eu/schema/3/d2Payload",
    "com": "http://datex2.eu/schema/3/common",
    "fac": "http://datex2.eu/schema/3/facilities",
    "loc": "http://datex2.eu/schema/3/locationReferencing",
    "en": "http://datex2.eu/schema/3/energyInfrastructure",
}

def text_or_none(elem, path, ns=NS):
    found = elem.find(path, ns)
    if found is not None and found.text is not None:
        txt = found.text.strip()
        return txt if txt != "" else None
    return None

def texts(elem, path, ns=NS):
    vals = []
    for found in elem.findall(path, ns):
        if found is not None and found.text is not None:
            txt = found.text.strip()
            if txt != "":
                vals.append(txt)
    return vals

def safe_float(x):
    if x is None:
        return None
    try:
        return float(str(x).replace(",", "."))
    except:
        return None

def strip_ns(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

# inspect connector child tags once, so we know where power lives
first_site = next(elem for elem in root.iter() if elem.tag.endswith("energyInfrastructureSite"))
first_connector = first_site.find(".//en:connector", NS)

print("Sample connector subtree tags:")
if first_connector is not None:
    seen = []
    for e in first_connector.iter():
        seen.append(strip_ns(e.tag))
    print(pd.Series(seen).value_counts())
else:
    print("No connector found in first site.")

Sample connector subtree tags:
connector           1
connectorType       1
chargingMode        1
connectorFormat     1
maxPowerAtSocket    1
voltage             1
maximumCurrent      1
Name: count, dtype: int64


In [9]:
def print_tree(elem, level=0, max_depth=4):
    if elem is None or level > max_depth:
        return
    indent = "  " * level
    text = f": {elem.text.strip()}" if elem.text and elem.text.strip() else ""
    print(f"{indent}{strip_ns(elem.tag)}{text}")
    for child in elem:
        print_tree(child, level + 1, max_depth)

print_tree(first_connector, max_depth=4)

connector
  connectorType: iec62196T2COMBO
  chargingMode: mode4DC
  connectorFormat: cableMode3
  maxPowerAtSocket: 350000.0
  voltage: 920.0
  maximumCurrent: 348.0


In [15]:
rows = []

for site in root.iter():
    if not site.tag.endswith("energyInfrastructureSite"):
        continue

    # --- basic fields ---
    site_name = text_or_none(site, ".//fac:name/com:values/com:value")
    site_type = text_or_none(site, ".//en:typeOfSite")
    last_updated = text_or_none(site, ".//fac:lastUpdated")

    operator_name = text_or_none(site, ".//fac:operator/fac:name/com:values/com:value")

    lat = safe_float(text_or_none(site, ".//loc:latitude"))
    lon = safe_float(text_or_none(site, ".//loc:longitude"))

    accessibility = text_or_none(site, ".//fac:accessibility")

    # --- connectors aggregation ---
    connectors = site.findall(".//en:connector", NS)

    n_connectors = 0
    powers = []

    for c in connectors:
        n_connectors += 1
        power = safe_float(text_or_none(c, ".//en:maxPowerAtSocket"))
        if power is not None:
            power = power / 1000  # convert W → kW
            powers.append(power)

    max_power = max(powers) if powers else None
    total_power = sum(powers) if powers else None

    # --- refill points ---
    refill_points = site.findall(".//en:refillPoint", NS)
    n_refill_points = len(refill_points)

    rows.append({
        "site_name": site_name,
        "operator": operator_name,
        "latitude": lat,
        "longitude": lon,
        "site_type": site_type,
        "accessibility": accessibility,
        "last_updated": last_updated,
        "n_refill_points": n_refill_points,
        "n_connectors": n_connectors,
        "max_power_kw": max_power,
        "total_power_kw": total_power,
        "source": "DGT_DATEX2"
    })

charging_df = pd.DataFrame(rows)

print("Total stations:", len(charging_df))
charging_df.head()

Total stations: 12066


,site_name,operator,latitude,longitude,site_type,accessibility,last_updated,n_refill_points,n_connectors,max_power_kw,total_power_kw,source
0,Centro Porsche Baleares CP2,Motor Box Mallorca SL,39.5948,2.635860,other,None,2023-11-21T11:39:04.000+01:00,1,1,350.0,350.0,DGT_DATEX2
1,Centro Porsche Baleares CP1,Motor Box Mallorca SL,39.5948,2.635900,other,None,2023-11-21T11:43:04.000+01:00,1,1,350.0,350.0,DGT_DATEX2
2,Centro Porsche Alicante estación de carga 2,Tuwyncar Sport S.L.,38.3466,-0.522950,other,None,2023-11-21T11:56:08.000+01:00,1,1,350.0,350.0,DGT_DATEX2
3,Centro Porsche Alicante estación de carga 1,Tuwyncar Sport S.L.,38.3460,-0.523464,other,None,2023-11-21T11:59:57.000+01:00,1,1,350.0,350.0,DGT_DATEX2
4,Centro Porsche Asturias CP 2,Ibérica de Tecnología y Servicios de Automoció...,43.3948,-5.816220,other,None,2023-11-21T16:36:28.000+01:00,1,1,320.0,320.0,DGT_DATEX2


In [16]:
charging_df.describe()

,latitude,longitude,n_refill_points,n_connectors,max_power_kw,total_power_kw
count,12066.000000,12066.000000,12066.000000,12066.000000,12066.000000,12066.000000
mean,39.845062,-2.672201,2.962208,3.540030,49.895255,158.286029
std,3.093529,4.107530,3.466933,4.376965,58.834189,335.089882
min,27.751438,-18.015152,1.000000,1.000000,0.060000,0.120000
25%,38.597065,-4.709173,1.000000,2.000000,22.000000,44.000000
50%,40.440478,-2.863155,2.000000,2.000000,22.170000,88.680000
75%,41.648758,0.023800,3.000000,4.000000,50.000000,143.650000
max,43.685726,4.293881,70.000000,160.000000,1000.000000,10800.000000


In [17]:
charging_df[['n_connectors', 'max_power_kw', 'total_power_kw']].head(20)

,n_connectors,max_power_kw,total_power_kw
0,1,350.0,350.0
1,1,350.0,350.0
2,1,350.0,350.0
3,1,350.0,350.0
4,1,320.0,320.0
5,1,320.0,320.0
6,1,320.0,320.0
7,1,320.0,320.0
8,1,320.0,320.0
9,1,320.0,320.0


In [18]:
charging_df[['latitude', 'longitude']].isna().mean()

,0
latitude,0.0
longitude,0.0


In [19]:
charging_df.duplicated(subset=['site_name', 'latitude', 'longitude']).sum()

np.int64(3)

In [20]:
charging_df = charging_df.drop_duplicates(
    subset=['site_name', 'latitude', 'longitude']
)

In [30]:
# convert to GeoDataFrame
import geopandas as gpd

mobility_gdf = gpd.read_file("mobility_backbone.gpkg")
print(mobility_gdf.crs)
print(mobility_gdf.head())

charging_gdf = gpd.GeoDataFrame(
    charging_df,
    geometry=gpd.points_from_xy(charging_df.longitude, charging_df.latitude),
    crs="EPSG:4326"
)

charging_gdf = charging_gdf.to_crs(mobility_gdf.crs)

EPSG:25830
                         id_catalog           provincia carretera  pk_inicio  \
0  022A0237768FDBE72758FA48460048FF    Alicante/Alacant      AP-7      621.1   
1  022A080D3C0558B82758FA48460047C9    Alicante/Alacant      AP-7      608.0   
2  022AA92FD6CAB6602758FA4846003637  Castellón/Castelló      AP-7      406.4   
3  022AAE88AF253C992758FA4846003543  Castellón/Castelló      AP-7      391.4   
4  0280076068BBED492758FA48460034DE  Castellón/Castelló      AP-7      365.6   

   pk_fin  longitud pkinicio_t    pkfin_t             inicio          fin  \
0   637.8    16.647  621+00100  637+00800         Enl. N-332   Enl. N-332   
1   621.1    13.058  608+00000  621+00100        Enl. CV-725   Enl. N-332   
2   425.5    19.003  406+00400  425+00500         Enl. N-340  Enl. N-340A   
3   406.4    14.924  391+00400  406+00400         Enl. N-340   Enl. N-340   
4   391.4    25.669  365+00600  391+00400  Enl. N-340/N-340A   Enl. N-340   

             clase  is_nacional  \
0  Autopis

In [31]:
mobility_gdf.crs

<Projected CRS: EPSG:25830>
Name: ETRS89 / UTM zone 30N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Europe between 6°W and 0°W: Faroe Islands offshore; Ireland - offshore; Jan Mayen - offshore; Norway including Svalbard - offshore; Spain - mainland - onshore and offshore.
- bounds: (-6.0, 35.26, 0.01, 80.49)
Coordinate Operation:
- name: UTM zone 30N
- method: Transverse Mercator
Datum: European Terrestrial Reference System 1989 ensemble
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [32]:
print(charging_gdf.geometry.head())

0     POINT (984054.036 4397985.95)
1    POINT (984057.473 4397986.166)
2     POINT (716466.49 4247176.228)
3    POINT (716423.353 4247108.433)
4    POINT (271924.395 4808510.696)
Name: geometry, dtype: geometry


In [34]:
# Spatial join: matching the charging stations for the mobility backbone (Which existing charging stations are actually relevant to the interurban network?)
import geopandas as gpd

def stations_near_backbone(charging_gdf, mobility_gdf, buffer_m):
    mobility_buffer = mobility_gdf[['geometry']].copy()
    mobility_buffer['geometry'] = mobility_buffer.buffer(buffer_m)

    joined = gpd.sjoin(
        charging_gdf,
        mobility_buffer,
        how='inner',
        predicate='intersects'
    )

    # remove possible duplicates if a station intersects multiple buffered segments
    joined = joined.drop_duplicates(subset=['site_name', 'latitude', 'longitude'])
    return joined

for dist in [500, 1000, 2000]:
    matched = stations_near_backbone(charging_gdf, mobility_gdf, dist)
    print(f"Buffer {dist} m -> {len(matched)} stations")

Buffer 500 m -> 2766 stations
Buffer 1000 m -> 4314 stations
Buffer 2000 m -> 6438 stations


In [37]:
# we keep 1000m as a baseline because:
# 2000 m is too permissive -> You’re including more than half of all stations. That contradicts the goal:
# identify interurban infrastructure, not everything

# 500 m is probably too strict -> Risks excluding: highway service areas slightly offset, industrial zones near highways,
# and rest stops not perfectly snapped to road geometry

# 1000 -> Keeps a meaningful subset (~35%), Includes realistic roadside infrastructure, Avoids most urban noise,
# aligns with typical spatial tolerance in infra planning


baseline_stations = stations_near_backbone(
    charging_gdf,
    mobility_gdf,
    1000
)

print("Baseline stations:", len(baseline_stations))

Baseline stations: 4314


In [38]:
baseline_stations.sample(10)[['site_name', 'latitude', 'longitude']]

,site_name,latitude,longitude
10852,"Repsol, GLP-Huelva",37.183784,-6.911489
9417,PETROCASH LUGO,43.030815,-7.536396
9523,San Fernando - Mercado Centro,36.466194,-6.199715
8366,A-NOVELDA-003,38.401530,-0.780560
11305,"Repsol ES, CRED Benahavi",36.473248,-5.010904
3515,VEPERSA PONTEVEDRA - CONCESIONARIO AUDI,42.440037,-8.618166
1676,ACCIONA - Cafeteria la Cadiera,41.311134,-2.001910
7800,Centro Comercial Altrium,41.681202,2.487215
3437,ES_BP_ISIDORO,40.482100,-3.396500
3463,Consum - Elche,38.261410,-0.712164


In [40]:
baseline_stations.to_file("baseline_stations.gpkg", driver="GPKG")
baseline_stations.drop(columns="geometry").to_csv("baseline_stations.csv", index=False)